# Đề xuất nghiên cứu mở rộng: Baseline + Hybrid SARIMAX–LSTM

Notebook chạy **trong repo**: đọc trực tiếp `data/processed/fa_data.csv` (sau khi chạy `main.py` hoặc `factor_analysis`), so sánh với pipeline chính (gộp ngày → `auto_arima` + exog Factor1–3, test 20%). Chạy từ thư mục gốc project hoặc `notebooks/` đều được.

**Nội dung:**
1. **SARIMAX (pipeline-aligned):** `pmdarima.auto_arima` giống `src/sarima_model.py` (mùa vụ tuần `m=7`).
2. **Baseline:** Random Forest, XGBoost trên **lag PM2.5** + **Factor1–3**.
3. **LSTM thuần:** một biến PM2.5, horizon 1 bước trên tập test.
4. **Hybrid (SARIMAX + LSTM trên phần dư):** SARIMAX nắm xu hướng/mùa/exog; LSTM học chuỗi **phần dư trong mẫu** rồi **dự báo lặp** phần dư trên test; tổng dự báo = dự báo SARIMAX + phần dư do LSTM dự báo.

In [ ]:
# Phụ thuộc: từ thư mục gốc repo chạy `pip install -r requirements.txt` (có pmdarima, tensorflow, xgboost, …).

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.models import Sequential

from pmdarima import auto_arima
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

np.random.seed(42)
tf.random.set_seed(42)

FACTOR_COLS = ["Factor1", "Factor2", "Factor3"]
TEST_RATIO = 0.2
SEASONAL_M = 7
LAG_TABULAR = 14
LSTM_LOOKBACK = 14
LSTM_EPOCHS = 50
LSTM_BATCH = 16

In [ ]:
def resolve_fa_data_path() -> Path:
    """Tìm fa_data.csv trong repo: data/processed/fa_data.csv hoặc fa_data.csv gần cwd."""
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "processed" / "fa_data.csv",
        cwd.parent / "data" / "processed" / "fa_data.csv",
        cwd / "fa_data.csv",
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    try:
        from google.colab import files

        print("Colab: chọn file fa_data.csv …")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Không có file được upload.")
        name = next(iter(uploaded.keys()))
        return Path(name).resolve()
    except ImportError:
        pass
    raise FileNotFoundError(
        "Không tìm thấy data/processed/fa_data.csv. Chạy `main.py` hoặc factor analysis trước, "
        "hoặc đặt fa_data.csv trong thư mục gốc repo / cùng thư mục khi chạy notebook."
    )


def load_fa_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.sort_index()
    return df


def aggregate_to_daily(df: pd.DataFrame) -> pd.DataFrame:
    """Giống src/sarima_model.aggregate_to_daily."""
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    daily = df[numeric_cols].resample("D").mean()
    return daily.dropna(how="all")


def train_test_split_ts(df: pd.DataFrame, test_ratio: float = TEST_RATIO):
    """Giống src/sarima_model.train_test_split."""
    n = len(df)
    split_idx = int(n * (1 - test_ratio))
    train, test = df.iloc[:split_idx], df.iloc[split_idx:]
    y_train, y_test = train["PM2.5"], test["PM2.5"]
    exog_cols = [c for c in FACTOR_COLS if c in df.columns]
    exog_train = train[exog_cols] if exog_cols else None
    exog_test = test[exog_cols] if exog_cols else None
    return y_train, y_test, exog_train, exog_test, split_idx


def compute_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    m = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[m], y_pred[m]
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    mape = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100)
    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}


p = resolve_fa_data_path()
print("Nguồn dữ liệu:", p)

raw = load_fa_csv(p)
daily = aggregate_to_daily(raw)
if "PM2.5" not in daily.columns:
    raise ValueError("Thiếu cột PM2.5 sau khi gộp ngày.")
daily = daily.dropna(subset=["PM2.5"])
for c in FACTOR_COLS:
    if c in daily.columns:
        daily[c] = daily[c].ffill().bfill()

y_train, y_test, exog_train, exog_test, split_idx = train_test_split_ts(daily)
print(f"Daily rows: {len(daily)} | Train: {len(y_train)} | Test: {len(y_test)} | split_idx={split_idx}")

## 1. SARIMAX (cùng logic pipeline: `auto_arima`, exog, m=7)

In [ ]:
X_tr = exog_train.values if exog_train is not None else None
X_te = exog_test.values if exog_test is not None else None

sarimax_model = auto_arima(
    y_train,
    X=X_tr,
    seasonal=True,
    m=SEASONAL_M,
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
    trace=False,
)
print("auto_arima:", sarimax_model.order, "x", sarimax_model.seasonal_order)

pred_sarimax = sarimax_model.predict(n_periods=len(y_test), X=X_te)
pred_sarimax = np.asarray(pred_sarimax, dtype=float).ravel()
metrics_sarimax = compute_metrics(y_test.values, pred_sarimax)
print("SARIMAX (test):", metrics_sarimax)

try:
    ins = sarimax_model.predict_in_sample(X=X_tr)
    ins = np.asarray(ins, dtype=float).ravel()
except Exception:
    ins = np.asarray(sarimax_model.arima_res_.fittedvalues, dtype=float).ravel()

if len(ins) != len(y_train):
    ins = ins[-len(y_train) :]
res_train = np.asarray(y_train.values, dtype=float).ravel() - ins
print("Phần dư train (SARIMAX): mean=", float(np.mean(res_train)), "std=", float(np.std(res_train)))

## 2. Baseline: Random Forest & XGBoost (lag PM2.5 + Factor1–3)

In [ ]:
def build_lag_tabular(pm25: pd.Series, factors: pd.DataFrame, lag: int, split_idx: int):
    pm = pm25.astype(float)
    rows_X, rows_y, rows_idx, rows_pos = [], [], [], []
    fac = factors.reindex(pm.index).ffill().bfill()
    vals = pm.values
    for t in range(lag, len(pm)):
        lags = vals[t - lag : t]
        fvec = fac.iloc[t][FACTOR_COLS].values.astype(float) if all(
            c in fac.columns for c in FACTOR_COLS
        ) else np.zeros(3, dtype=float)
        rows_X.append(np.concatenate([lags, fvec]))
        rows_y.append(vals[t])
        rows_idx.append(pm.index[t])
        rows_pos.append(t)
    X = np.asarray(rows_X, dtype=float)
    y = np.asarray(rows_y, dtype=float)
    idx = pd.DatetimeIndex(rows_idx)
    pos = np.asarray(rows_pos, dtype=int)
    train_mask = pos < split_idx
    test_mask = ~train_mask
    return X, y, idx, train_mask, test_mask


pm_all = daily["PM2.5"]
fac_all = daily[FACTOR_COLS] if all(c in daily.columns for c in FACTOR_COLS) else pd.DataFrame(index=daily.index)
X_tab, y_tab, idx_tab, tr_m, te_m = build_lag_tabular(pm_all, fac_all, LAG_TABULAR, split_idx)

rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_tab[tr_m], y_tab[tr_m])
pred_rf = rf.predict(X_tab[te_m])
metrics_rf = compute_metrics(y_tab[te_m], pred_rf)

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb.fit(X_tab[tr_m], y_tab[tr_m])
pred_xgb = xgb.predict(X_tab[te_m])
metrics_xgb = compute_metrics(y_tab[te_m], pred_xgb)

print("RandomForest:", metrics_rf)
print("XGBoost:", metrics_xgb)

## 3. LSTM thuần (univariate PM2.5, giống hướng LSTM.ipynb)

In [ ]:
def build_sequences(arr_1d: np.ndarray, look_back: int):
    x, y = [], []
    for i in range(len(arr_1d) - look_back):
        x.append(arr_1d[i : i + look_back])
        y.append(arr_1d[i + look_back])
    return np.asarray(x, dtype=float), np.asarray(y, dtype=float)


values = pm_all.values.reshape(-1, 1).astype(float)
sp = split_idx
lb = LSTM_LOOKBACK
train_vals = values[:sp]
test_concat = values[sp - lb :]

scaler_y = MinMaxScaler((0, 1))
train_s = scaler_y.fit_transform(train_vals).ravel()
test_s = scaler_y.transform(test_concat).ravel()

x_tr, y_tr = build_sequences(train_s, lb)
x_te, y_te = build_sequences(test_s, lb)
x_tr = x_tr.reshape(-1, lb, 1)
x_te = x_te.reshape(-1, lb, 1)

lstm_uni = Sequential(
    [
        LSTM(32, input_shape=(lb, 1)),
        Dense(1),
    ]
)
lstm_uni.compile(optimizer="adam", loss="mse")
lstm_uni.fit(x_tr, y_tr, epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH, verbose=0)

pred_s = lstm_uni.predict(x_te, verbose=0).ravel()
pred_lstm = scaler_y.inverse_transform(pred_s.reshape(-1, 1)).ravel()
y_true_lstm = scaler_y.inverse_transform(y_te.reshape(-1, 1)).ravel()
pred_lstm = pred_lstm[: len(y_test)]
y_true_lstm = y_true_lstm[: len(y_test)]
metrics_lstm = compute_metrics(y_true_lstm, pred_lstm)
print("LSTM (uni, test):", metrics_lstm)

## 4. Hybrid: SARIMAX (test) + LSTM học & dự báo **phần dư** (rolling trên test)

LSTM được huấn luyện trên chuỗi phần dư trong mẫu `y_train - fitted_in_sample`; trên test, cửa sổ phần dư được cập nhật bằng dự báo phần dư từng bước, cộng vào `pred_sarimax`.

In [ ]:
scaler_res = MinMaxScaler((0, 1))
res_scaled = scaler_res.fit_transform(res_train.reshape(-1, 1)).ravel()
rx, ry = build_sequences(res_scaled, lb)
if len(rx) < 5:
    raise ValueError("Không đủ điểm để huấn luyện LSTM trên phần dư.")
rx = rx.reshape(-1, lb, 1)

lstm_res = Sequential(
    [
        LSTM(32, input_shape=(lb, 1)),
        Dense(1),
    ]
)
lstm_res.compile(optimizer="adam", loss="mse")
lstm_res.fit(rx, ry, epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH, verbose=0)

window = list(res_scaled[-lb:])
residual_preds_scaled = []
for _ in range(len(y_test)):
    x_in = np.asarray(window, dtype=float).reshape(1, lb, 1)
    r_next = float(lstm_res.predict(x_in, verbose=0).ravel()[0])
    residual_preds_scaled.append(r_next)
    window = window[1:] + [r_next]

residual_hat = scaler_res.inverse_transform(np.asarray(residual_preds_scaled).reshape(-1, 1)).ravel()
pred_hybrid = pred_sarimax + residual_hat
metrics_hybrid = compute_metrics(y_test.values, pred_hybrid)
print("Hybrid SARIMAX + LSTM(residual):", metrics_hybrid)

## 5. Bảng so sánh & biểu đồ (có thể đối chiếu `reports/tables/evaluation_metrics.csv` nếu đã chạy pipeline đánh giá)

In [ ]:
summary = pd.DataFrame(
    [
        {"Model": "SARIMAX (auto_arima, pipeline-aligned)", **metrics_sarimax},
        {"Model": "RandomForest (lags+FA)", **metrics_rf},
        {"Model": "XGBoost (lags+FA)", **metrics_xgb},
        {"Model": "LSTM univariate", **metrics_lstm},
        {"Model": "Hybrid SARIMAX + LSTM(residual)", **metrics_hybrid},
    ]
).sort_values("RMSE")
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(summary["Model"], summary["RMSE"], color="#2E86AB")
ax.set_ylabel("RMSE")
ax.set_title("So sánh RMSE trên tập test (20% cuối, daily)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(12, 4))
ax2.plot(y_test.index, y_test.values, label="Actual", color="#333")
ax2.plot(y_test.index, pred_sarimax, label="SARIMAX", alpha=0.85)
ax2.plot(y_test.index, pred_hybrid, label="Hybrid", alpha=0.85)
ax2.legend()
ax2.set_title("PM2.5 daily — Actual vs SARIMAX vs Hybrid")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()